# Semaine 14 — Modèles génératifs modernes : démonstrations
**Apprentissage automatique I (MATH 60629)** — Laurent Charlin

Ce carnet contient les quatre démonstrations du cours. Chaque section est indépendante.

| Démo | Idée | Matériel |
|---|---|---|
| A | Un GMM est déjà un modèle génératif : *évaluer* et *échantillonner* | CPU |
| B | Un modèle de diffusion en 2D, de zéro | CPU (~1 min) |
| C | CLIP : relier images et textes | CPU ou GPU |
| D | Texte → image et le *guidage* (classifier-free guidance) | **GPU** (Exécution → Modifier le type d'exécution → GPU T4) |

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

rng = np.random.default_rng(0)

# Données jouet utilisées dans les démos A et B : deux « lunes » en 2D
X, _ = make_moons(n_samples=5000, noise=0.05, random_state=0)
X = (X - X.mean(0)) / X.std(0)          # centrer-réduire

def cadre(ax, titre):
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(titre)

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(*X[:2000].T, s=3); cadre(ax, "Nos données")
plt.show()

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'sklearn'

---
## Démo A — Vous avez déjà construit un modèle génératif

À la semaine 8, nous avons utilisé des mélanges de gaussiennes (GMM) pour le **regroupement**.
Un GMM définit une distribution $P(x)$. On peut donc s'en servir pour :

1. **Évaluer** : quelle est la (log-)vraisemblance d'un point ? → `score_samples`
2. **Échantillonner** : produire de *nouveaux* points → `sample`

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=8, random_state=0).fit(X)
nouveaux, _ = gmm.sample(2000)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].scatter(*X[:2000].T, s=3); cadre(axes[0], "Données réelles")
axes[1].scatter(*nouveaux.T, s=3, c="C1"); cadre(axes[1], "Échantillons du GMM")
plt.show()

In [ ]:
# 1) Évaluer : un point « sur la lune » vs un point « dans le vide »
points = np.array([[0.0, 1.4],     # sur la lune du haut
                   [0.0, 0.0]])    # entre les deux lunes
print("log P(x) :", gmm.score_samples(points).round(2))

**À essayer :** changez `n_components` (2, 8, 50). Que se passe-t-il ?

**À retenir :** pour des points en 2D, un GMM suffit. Pour une image de 256×256×3 ≈ 200 000 nombres très dépendants les uns des autres, il nous faut des réseaux de neurones. C'est l'objet du reste du cours.

---
## Démo B — Un modèle de diffusion en 2D, de zéro

Idée en trois phrases :
1. **Détruire est facile** : on ajoute du bruit gaussien, petit à petit, jusqu'à obtenir du bruit pur.
2. **On apprend à défaire** : un réseau reçoit un point bruité $x_t$ et l'étape $t$, et **prédit le bruit** ajouté. C'est une simple **régression** (perte quadratique).
3. **Générer** : on part du bruit pur et on retire, étape par étape, le bruit prédit.

Le bruitage a une forme fermée : $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$, avec $\varepsilon \sim \mathcal N(0, I)$.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

T = 100                                        # nombre d'étapes de bruitage
betas = torch.linspace(1e-4, 0.1, T)           # quantité de bruit ajoutée à chaque étape
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)       # combien du signal original il reste à l'étape t

X_t = torch.tensor(X, dtype=torch.float32)

def bruiter(x0, t, eps):
    # x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * eps
    ab = alpha_bar[t].unsqueeze(1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps

# Le processus « avant » : on regarde les données se dissoudre
etapes = [0, 10, 25, 50, 99]
fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, t in zip(axes, etapes):
    tt = torch.full((2000,), t)
    xt = bruiter(X_t[:2000], tt, torch.randn(2000, 2))
    ax.scatter(*xt.T, s=2); cadre(ax, f"t = {t}")
plt.suptitle("Processus avant : données → bruit"); plt.show()

In [ ]:
# Le réseau « débruiteur » : un petit MLP qui reçoit (x_t, t) et prédit le bruit epsilon
class Debruiteur(nn.Module):
    def __init__(self, h=128, n_freq=8):
        super().__init__()
        self.freq = torch.arange(1, n_freq + 1) * torch.pi
        self.net = nn.Sequential(
            nn.Linear(2 + 2 * n_freq, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, 2))
    def forward(self, x, t):
        # on encode l'étape t avec des sinus/cosinus (comme l'encodage de position des transformers)
        tt = (t.float() / T).unsqueeze(1) * self.freq
        return self.net(torch.cat([x, tt.sin(), tt.cos()], dim=1))

modele = Debruiteur()
opt = torch.optim.Adam(modele.parameters(), lr=1e-3)

# Entraînement : une régression ordinaire !
pertes = []
for it in range(4000):
    x0 = X_t[torch.randint(0, len(X_t), (512,))]
    t = torch.randint(0, T, (512,))
    eps = torch.randn_like(x0)
    perte = ((modele(bruiter(x0, t, eps), t) - eps) ** 2).mean()   # || eps - eps_prédit ||^2
    opt.zero_grad(); perte.backward(); opt.step()
    pertes.append(perte.item())

plt.plot(np.convolve(pertes, np.ones(50) / 50, mode="valid"))
plt.xlabel("itération"); plt.ylabel("perte (moyenne mobile)"); plt.show()

In [ ]:
# Génération : on part du bruit pur et on débruite étape par étape
@torch.no_grad()
def generer(n=2000, garder=(99, 50, 25, 10, 0)):
    x = torch.randn(n, 2)
    instantanes = {T - 1: x.clone()}
    for t in reversed(range(T)):
        tt = torch.full((n,), t)
        eps_pred = modele(x, tt)
        # retirer la part de bruit prédite
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps_pred) / alphas[t].sqrt()
        if t > 0:                                   # un peu de hasard à chaque étape
            x = x + betas[t].sqrt() * torch.randn_like(x)
        if t in garder:
            instantanes[t] = x.clone()
    return instantanes

inst = generer()
fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, t in zip(axes, [99, 50, 25, 10, 0]):
    ax.scatter(*inst[t].T, s=2, c="C1"); cadre(ax, f"t = {t}")
plt.suptitle("Processus inverse : bruit → données"); plt.show()

In [ ]:
# (Optionnel) Animation du processus inverse
from matplotlib import animation
from IPython.display import HTML

@torch.no_grad()
def trajectoire(n=1500):
    x = torch.randn(n, 2); traj = [x.clone()]
    for t in reversed(range(T)):
        eps_pred = modele(x, torch.full((n,), t))
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps_pred) / alphas[t].sqrt()
        if t > 0: x = x + betas[t].sqrt() * torch.randn_like(x)
        traj.append(x.clone())
    return traj

traj = trajectoire()
fig, ax = plt.subplots(figsize=(4.5, 4.5)); sc = ax.scatter(*traj[0].T, s=2, c="C1")
def maj(i):
    sc.set_offsets(traj[i].numpy()); cadre(ax, f"étape {i} / {T}"); return (sc,)
anim = animation.FuncAnimation(fig, maj, frames=len(traj), interval=60)
plt.close(fig); HTML(anim.to_jshtml())

**À essayer :**
- Réduisez `T` à 10. La génération est-elle aussi bonne ? (Peu de grandes étapes, c'est plus difficile que beaucoup de petites.)
- Remplacez `make_moons` par une autre forme (p. ex. `sklearn.datasets.make_circles`) et réentraînez.

**À retenir :** le réseau n'a jamais vu « une lune ». Il a seulement appris à retirer du bruit. Les modèles qui génèrent des images font *exactement* cela, avec un plus gros réseau (U-Net ou transformer) et des images au lieu de points.

---
## Démo C — CLIP : relier images et textes

CLIP apprend deux encodeurs (un pour les images, un pour les textes) de sorte que l'image et sa légende aient des représentations **similaires**, et que les paires qui ne vont pas ensemble soient **dissimilaires** (apprentissage contrastif).

Ci-dessous, on calcule la matrice de similarité entre quelques images et quelques légendes. Remarque : le CLIP original a été entraîné sur des légendes **en anglais**.

In [ ]:
!pip -q install transformers pillow requests

In [ ]:
import requests, torch
from PIL import Image
from io import BytesIO
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Quelques images du jeu de données COCO (remplacez-les par les vôtres au besoin)
urls = {
    "chats":         "http://images.cocodataset.org/val2017/000000039769.jpg",
    "ours":          "http://images.cocodataset.org/val2017/000000000285.jpg",
    "arrêt":         "http://images.cocodataset.org/val2017/000000000724.jpg",
    "oursons":       "http://images.cocodataset.org/val2017/000000000776.jpg",
}
images = [Image.open(BytesIO(requests.get(u, timeout=30).content)).convert("RGB") for u in urls.values()]

legendes = ["two cats sleeping on a couch",
            "a brown bear in the grass",
            "a stop sign on a street",
            "a pile of teddy bears"]

fig, axes = plt.subplots(1, len(images), figsize=(14, 3.5))
for ax, im, nom in zip(axes, images, urls):
    ax.imshow(im); ax.set_title(nom); ax.axis("off")
plt.show()

In [ ]:
with torch.no_grad():
    entree = proc(text=legendes, images=images, return_tensors="pt", padding=True)
    sortie = clip(**entree)
    I = sortie.image_embeds / sortie.image_embeds.norm(dim=-1, keepdim=True)
    Tx = sortie.text_embeds / sortie.text_embeds.norm(dim=-1, keepdim=True)
    sim = (I @ Tx.T).numpy()          # similarité cosinus image_i · texte_j

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(sim, cmap="Blues")
ax.set_xticks(range(len(legendes)), legendes, rotation=30, ha="right")
ax.set_yticks(range(len(images)), list(urls))
for i in range(sim.shape[0]):
    for j in range(sim.shape[1]):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center")
ax.set_title("Similarité image × texte (CLIP)"); plt.colorbar(im); plt.show()

**À essayer :** ajoutez une légende piège (p. ex. `"two dogs sleeping on a couch"`) ou une légende en français. Que devient la diagonale ?

**À retenir :** CLIP place textes et images dans un **même espace**. C'est ce qui permet ensuite de *conditionner* la génération d'images sur un texte.

---
## Démo D — Texte → image et le guidage (*classifier-free guidance*)

⚠️ **Nécessite un GPU.** (Exécution → Modifier le type d'exécution → GPU T4.) Le premier téléchargement du modèle prend quelques minutes.

On utilise Stable Diffusion 1.5, un modèle de **diffusion latente** : le débruitage se fait dans un espace compressé par un auto-encodeur, puis le décodeur produit l'image.

Le **guidage** $w$ est un bouton : le modèle prédit le bruit *avec* et *sans* le texte, et on pousse dans la direction du texte :
$\hat\varepsilon = \varepsilon_\theta(x_t, \varnothing) + w\,\big(\varepsilon_\theta(x_t, \text{texte}) - \varepsilon_\theta(x_t, \varnothing)\big)$

In [ ]:
!pip -q install diffusers transformers accelerate

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16
).to("cuda")
pipe.set_progress_bar_config(disable=True)

In [ ]:
prompt = "a watercolor painting of a fox reading a book in a library"   # en anglais
graine = 0
guidages = [1.0, 3.0, 7.5, 15.0]      # w = 1 : pas de guidage

images = []
for w in guidages:
    g = torch.Generator("cuda").manual_seed(graine)   # même graine → même bruit de départ
    images.append(pipe(prompt, guidance_scale=w, num_inference_steps=30, generator=g).images[0])

fig, axes = plt.subplots(1, len(images), figsize=(16, 4.5))
for ax, im, w in zip(axes, images, guidages):
    ax.imshow(im); ax.set_title(f"guidage w = {w}"); ax.axis("off")
plt.show()

**Questions :** qu'est-ce qui change quand $w$ augmente ? L'image respecte-t-elle mieux le texte ? Est-elle plus variée ou moins variée ? Plus naturelle ou plus « saturée » ?

In [ ]:
# Voir le débruitage : on décode l'image latente à quelques étapes intermédiaires
intermediaires = []
def rappel(pipe, i, t, kw):
    if i % 6 == 0 or i == 29:
        with torch.no_grad():
            lat = kw["latents"] / pipe.vae.config.scaling_factor
            img = pipe.vae.decode(lat).sample
        img = ((img[0].float().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()
        intermediaires.append((i, img))
    return kw

g = torch.Generator("cuda").manual_seed(graine)
pipe(prompt, guidance_scale=7.5, num_inference_steps=30, generator=g,
     callback_on_step_end=rappel, callback_on_step_end_tensor_inputs=["latents"])

fig, axes = plt.subplots(1, len(intermediaires), figsize=(18, 3.6))
for ax, (i, img) in zip(axes, intermediaires):
    ax.imshow(img); ax.set_title(f"étape {i + 1}"); ax.axis("off")
plt.show()

**À essayer :** gardez la même graine et changez un seul mot de la consigne. Puis gardez la consigne et changez la graine.

**À retenir :** le texte n'est qu'une *condition* donnée au débruiteur ; le guidage règle le compromis **fidélité au texte ↔ diversité**.